# PLUTO — Comparativa de Métricas por Experimento

Genera una figura multi-panel limpia y exportable como **SVG** para insertar en Google Docs.  
Fuente de datos: `research/evaluation/experiment_results.json` + corrección Exp_05 según README.

---

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import numpy as np

# ── Rutas ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
JSON_PATH    = NOTEBOOK_DIR.parent / "evaluation" / "experiment_results.json"
SVG_OUT      = NOTEBOOK_DIR.parent / "evaluation" / "experiment_metrics.svg"

# ── Carga de datos ────────────────────────────────────────────────────────────
with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)

# Corrección Exp_05: ok_recall = 0.39 según README (la v2 del modelo)
raw["Exp_05"]["ok_recall"] = 0.39

experiments = list(raw.keys())          # ["Exp_01", ..., "Exp_11"]
labels = [e.replace("_", "\n") for e in experiments]   # salto de línea para el eje X

print(f"Experimentos cargados: {experiments}")
print(f"SVG de salida: {SVG_OUT}")

In [ ]:
# ── Métricas a visualizar ────────────────────────────────────────────────────
# Cada entrada: (clave JSON, título panel, si es el panel destacado)
METRICS = [
    ("f1_macro",    "F1-Macro",      False),
    ("f1_weighted", "F1-Weighted",   False),
    ("ok_recall",   "Recall OK ★",   True),   # panel destacado
    ("nok_recall",  "Recall NOK",    False),
    ("ok_f1",       "F1 clase OK",   False),
    ("nok_f1",      "F1 clase NOK",  False),
]

def get_values(key):
    return [raw[e][key] for e in experiments]

# ── Paleta de colores ────────────────────────────────────────────────────────
BG_FIGURE   = "#0c1629"     # fondo figura
BG_AX       = "#111c30"     # fondo ejes normales
BG_AX_HL   = "#0d1f3c"     # fondo eje destacado (ligeramente diferente)
BORDER_HL   = "#fbbf24"     # borde dorado para el panel Recall OK
BORDER_NORM = "#1e2d45"     # borde normal

COLOR_BAR   = "#2563eb"     # azul principal
COLOR_EXP5  = "#34d399"     # verde para Exp_05
COLOR_HL_BAR= "#fbbf24"     # dorado para barras en panel destacado
COLOR_HL_E5 = "#f87171"     # rojo para Exp_05 en panel destacado (máximo)

TEXT_PRIMARY   = "#f8fafc"
TEXT_SECONDARY = "#94a3b8"
GRID_COLOR     = "#1e2d45"

EXP5_IDX = experiments.index("Exp_05")

In [ ]:
# ── Construcción de la figura ─────────────────────────────────────────────────
N_METRICS = len(METRICS)
COLS = 3
ROWS = (N_METRICS + COLS - 1) // COLS     # 2 filas de 3

fig, axes = plt.subplots(
    ROWS, COLS,
    figsize=(16, 9),
    facecolor=BG_FIGURE,
)
fig.subplots_adjust(hspace=0.52, wspace=0.28, top=0.88, bottom=0.10, left=0.06, right=0.97)

# ── Título global ─────────────────────────────────────────────────────────────
fig.text(
    0.5, 0.96,
    "PLUTO — Comparativa de Experimentos de Clasificación (CTG GT14)",
    ha="center", va="top",
    fontsize=15, fontweight="bold",
    color=TEXT_PRIMARY,
    fontfamily="DejaVu Sans",
)
fig.text(
    0.5, 0.916,
    "Dataset: Dataset_01_Anonimizado.xlsx · Holdout 20% estratificado · Métrica prioritaria CTAG: Recall OK",
    ha="center", va="top",
    fontsize=8.5,
    color=TEXT_SECONDARY,
)

ax_flat = axes.flatten()
x = np.arange(len(experiments))

for idx, (key, title, is_highlight) in enumerate(METRICS):
    ax = ax_flat[idx]
    values = get_values(key)

    # ── Estilo del panel ──────────────────────────────────────────────────────
    bg     = BG_AX_HL if is_highlight else BG_AX
    border = BORDER_HL if is_highlight else BORDER_NORM
    lw     = 1.8 if is_highlight else 0.8

    ax.set_facecolor(bg)
    for spine in ax.spines.values():
        spine.set_edgecolor(border)
        spine.set_linewidth(lw)

    # ── Colores de cada barra ─────────────────────────────────────────────────
    if is_highlight:
        bar_colors = [COLOR_HL_E5 if i == EXP5_IDX else COLOR_HL_BAR for i in range(len(experiments))]
        alphas     = [1.0 if i == EXP5_IDX else 0.55 for i in range(len(experiments))]
    else:
        bar_colors = [COLOR_EXP5 if i == EXP5_IDX else COLOR_BAR for i in range(len(experiments))]
        alphas     = [1.0 if i == EXP5_IDX else 0.65 for i in range(len(experiments))]

    bars = ax.bar(
        x, values,
        color=bar_colors, alpha=1.0,
        width=0.62, zorder=3,
        linewidth=0,
    )
    # Aplicar alpha individualmente
    for bar, a in zip(bars, alphas):
        bar.set_alpha(a)

    # ── Etiqueta de valor sobre cada barra ───────────────────────────────────
    for i, (bar, val) in enumerate(zip(bars, values)):
        color = TEXT_PRIMARY if (i == EXP5_IDX) else TEXT_SECONDARY
        fw    = "bold" if i == EXP5_IDX else "normal"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            val + 0.008,
            f"{val:.2f}",
            ha="center", va="bottom",
            fontsize=6.2, color=color, fontweight=fw,
            zorder=4,
        )

    # ── Grid y eje Y ─────────────────────────────────────────────────────────
    ax.set_ylim(0, min(1.0, max(values) * 1.28))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(0.2))
    ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.1))
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.6, linestyle="--", zorder=0)
    ax.grid(axis="y", which="minor", color=GRID_COLOR, linewidth=0.3, linestyle=":", zorder=0)
    ax.tick_params(axis="y", colors=TEXT_SECONDARY, labelsize=7)

    # ── Eje X ────────────────────────────────────────────────────────────────
    ax.set_xticks(x)
    ax.set_xticklabels(
        labels, fontsize=6.8, color=TEXT_SECONDARY, linespacing=1.3,
    )
    ax.tick_params(axis="x", colors=TEXT_SECONDARY, length=0)

    # ── Resalte visual Exp_05 en el panel destacado ──────────────────────────
    if is_highlight:
        ax.axvline(x=EXP5_IDX, color=COLOR_HL_E5, linewidth=0.6, linestyle="--", alpha=0.4, zorder=2)
        max_val = max(values)
        ax.annotate(
            f" MÁXIMO\n {max_val:.2f}",
            xy=(EXP5_IDX, max_val),
            xytext=(EXP5_IDX + 0.55, max_val - 0.05),
            fontsize=6.5, color=COLOR_HL_E5, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=COLOR_HL_E5, lw=0.8),
            zorder=5,
        )

    # ── Título del panel ─────────────────────────────────────────────────────
    title_color = BORDER_HL if is_highlight else TEXT_PRIMARY
    title_fw    = "bold" if is_highlight else "semibold"
    ax.set_title(
        title,
        color=title_color, fontsize=9.5, fontweight=title_fw,
        pad=6,
    )

# ── Leyenda global ────────────────────────────────────────────────────────────
leg_patches = [
    mpatches.Patch(facecolor=COLOR_EXP5,   label="Exp_05 — VAE + CatBoost v2 (producción)"),
    mpatches.Patch(facecolor=COLOR_BAR,    label="Resto de experimentos", alpha=0.65),
    mpatches.Patch(facecolor=BORDER_HL,    label="Panel prioritario CTAG: Recall OK"),
]
fig.legend(
    handles=leg_patches,
    loc="lower center",
    ncol=3,
    bbox_to_anchor=(0.5, 0.005),
    frameon=True,
    framealpha=0.15,
    facecolor=BG_AX,
    edgecolor=BORDER_NORM,
    labelcolor=TEXT_SECONDARY,
    fontsize=8,
)

plt.savefig(
    SVG_OUT,
    format="svg",
    bbox_inches="tight",
    facecolor=BG_FIGURE,
    dpi=150,
)

plt.show()
print(f"\n✓ SVG guardado en: {SVG_OUT}")

---
## Instrucciones para insertar en Google Docs

1. Abre el archivo `research/evaluation/experiment_metrics.svg` con cualquier navegador o Inkscape para verificar que se ve correctamente.
2. En Google Docs: **Insertar → Imagen → Subir desde el equipo** → selecciona el `.svg`.  
   *(Google Docs acepta SVG directamente y lo mantiene vectorial)*
3. Ajusta el tamaño con las asas de la imagen para que ocupe todo el ancho de la página.

> **Nota**: si Google Docs no renderiza el fondo oscuro correctamente, también puedes exportar desde el notebook cambiando `format="svg"` a `format="png"` con `dpi=300` para obtener un PNG de alta resolución.